# ThingsBoard Full Harvest v9 — All Banks, All Keys, ML Export

## All 12 Bank Hierarchies (hardcoded, exact):
```
Bank of India        : Tenant → Customer → HO → NBG/FGMO → ZO → Branch
Bank of Baroda       : Tenant → Customer → HO → ZO → RO → Branch
Canara Bank          : Tenant → Customer → HO → RO → Branch
Bank of Maharashtra  : Tenant → Customer → HO → ZO → Branch
Central Bank of India: Tenant → Customer → Corporate Office → ZO → RO → Branch
Indian Bank          : Tenant → Customer → HO → ZO → Branch
Indian Overseas Bank : Tenant → Customer → HO → RO → Branch
Punjab & Sind Bank   : Tenant → Customer → HO → ZO → Branch
Punjab National Bank : Tenant → Customer → HO → ZO → CO → Branch
State Bank of India  : Tenant → Customer → HO → LHO → ZO → RBO → Branch
UCO Bank             : Tenant → Customer → HO → ZO → Branch
Union Bank of India  : Tenant → Customer → Central Office → ZO → RO → Branch
```

## Key improvements over v8:
| Feature | v8 | v9 |
|---|---|---|
| Bank hierarchy depth | generic | exact per-bank (2–6 levels) |
| Asset type mapping | guessed | exact: HO/NBG/FGMO/LHO/RBO/CO/RO/ZO |
| New keys from v8 discovery | ❌ | ✅ (285 new keys added) |
| Integrated Alarm System | ❌ | ✅ |
| usage_daily / usage_last_7/15 | ❌ | ✅ |
| BATTERY REVERSE / ON events | ❌ | ✅ |
| Full hierarchy path column | ❌ | ✅ |
| Dashboard JSON export | ❌ | ✅ |

> **Run cells 1–12 top to bottom. Set `.env` before Cell 3.**

---
## Cell 1 — Environment Setup

In [15]:
import pathlib
ENV = pathlib.Path('.env')
if not ENV.exists():
    ENV.write_text(
        'TB_HOST=https://seple.iot-private.cloud\n'
        'TB_EMAIL=info@seple.in\n'
        'TB_PASSWORD=yourpassword\n'
    )
    print('✅ .env created — fill TB_PASSWORD before continuing.')
else:
    print('✅ .env exists.')
print('   Add .env to .gitignore — never commit secrets.')


✅ .env exists.
   Add .env to .gitignore — never commit secrets.


---
## Cell 2 — Install Dependencies

In [16]:
import subprocess, sys
for p in ['requests','pandas','openpyxl','tqdm','urllib3','python-dotenv']:
    subprocess.check_call([sys.executable,'-m','pip','install',p,'-q'])
print('✅ All packages ready.')


✅ All packages ready.


---
## Cell 3 — Config + Auth

### Per-bank hierarchy definitions (exact, from your spec)
Level labels: `ho` → `nbg/fgmo` → `lho` → `zo` → `ro/co/rbo` → `branch`
Unknown asset types fall into the closest matching level.

In [17]:
import requests, json, time, warnings, os
from datetime import datetime
from collections import Counter, defaultdict
from dotenv import load_dotenv

load_dotenv(override=True)
warnings.filterwarnings('ignore')

TB_HOST     = os.environ['TB_HOST']
TB_EMAIL    = os.environ['TB_EMAIL']
TB_PASSWORD = os.environ['TB_PASSWORD']

PAGE_SIZE          = 100
REQUEST_DELAY      = 0.05
MAX_RELATION_DEPTH = 8     # SBI goes 6 levels deep
GAP_FAULT_DAYS     = 3

# ══════════════════════════════════════════════════════════════════════════════
# PER-BANK HIERARCHY MAP
# Each bank's asset types mapped to canonical level names.
# Levels (top→bottom): ho, nbg, lho, zo, ro, co, rbo, branch
# ══════════════════════════════════════════════════════════════════════════════
BANK_HIERARCHY = {
    # ── Bank of India ─────────────────────────────────────────────────────────
    # Tenant → Customer(BOI) → HO → NBG/FGMO → ZO → Branch
    'BANK OF INDIA': {
        'depth': 5,
        'levels': ['ho','nbg','zo','branch'],
        'type_map': {
            'Head Office BOI':   'ho',
            'NBG BOI':           'nbg',
            'Zonal Office BOI':  'zo',
            'Branch BOI':        'branch',
            'HO':  'ho', 'NBG': 'nbg', 'FGMO': 'nbg', 'ZO': 'zo',
        },
    },
    # ── Bank of Baroda ────────────────────────────────────────────────────────
    # Tenant → Customer(BOB) → HO → ZO → RO → Branch
    'BANK OF BARODA': {
        'depth': 5,
        'levels': ['ho','zo','ro','branch'],
        'type_map': {
            'Head Office BOB':   'ho',
            'Zonal Office BOB':  'zo',
            'Regional Office BOB':'ro',
            'Branch BOB':        'branch',
            'HO': 'ho', 'ZO': 'zo', 'RO': 'ro',
        },
    },
    # ── Canara Bank ───────────────────────────────────────────────────────────
    # Tenant → Customer(CB) → HO → RO → Branch
    'CANARA BANK': {
        'depth': 4,
        'levels': ['ho','ro','branch'],
        'type_map': {
            'Head Office CB':    'ho',
            'Regional Office CB':'ro',
            'Branch CB':         'branch',
            'HO': 'ho', 'RO': 'ro',
        },
    },
    # ── Bank of Maharashtra ───────────────────────────────────────────────────
    # Tenant → Customer → HO → ZO → Branch
    'BANK OF MAHARASHTRA': {
        'depth': 4,
        'levels': ['ho','zo','branch'],
        'type_map': {
            'Bank-Head Office':  'ho',
            'Bank-Zonal Office': 'zo',
            'Bank-Branch':       'branch',
            'HO': 'ho', 'ZO': 'zo',
        },
    },
    # ── Central Bank of India ─────────────────────────────────────────────────
    # Tenant → Customer → Corporate Office → ZO → RO → Branch
    'CENTRAL BANK OF INDIA': {
        'depth': 5,
        'levels': ['ho','zo','ro','branch'],
        'type_map': {
            'Corporate Office':  'ho',
            'ZO':                'zo',
            'RO':                'ro',
            'Bank-Branch':       'branch',
        },
    },
    # ── Indian Bank ───────────────────────────────────────────────────────────
    # Tenant → Customer → HO → ZO → Branch
    'INDIAN BANK': {
        'depth': 4,
        'levels': ['ho','zo','branch'],
        'type_map': {'HO':'ho','ZO':'zo','Branch':'branch'},
    },
    # ── Indian Overseas Bank ──────────────────────────────────────────────────
    # Tenant → Customer → HO → RO → Branch
    'INDIAN OVERSEAS BANK': {
        'depth': 4,
        'levels': ['ho','ro','branch'],
        'type_map': {'HO':'ho','RO':'ro','Branch':'branch'},
    },
    # ── Punjab & Sind Bank ────────────────────────────────────────────────────
    # Tenant → Customer → HO → ZO → Branch
    'PANJAB & SIND BANK': {
        'depth': 4,
        'levels': ['ho','zo','branch'],
        'type_map': {'HO':'ho','ZO':'zo','Branch':'branch'},
    },
    # ── Punjab National Bank ──────────────────────────────────────────────────
    # Tenant → Customer → HO → ZO → CO → Branch
    'PUNJAB NATIONAL BANK': {
        'depth': 5,
        'levels': ['ho','zo','co','branch'],
        'type_map': {
            'HO':'ho','ZO':'zo','CO':'co','Circle Office':'co','Branch':'branch',
        },
    },
    # ── State Bank of India ───────────────────────────────────────────────────
    # Tenant → Customer → HO → LHO → ZO → RBO → Branch  (6 levels!)
    'STATE BANK OF INDIA': {
        'depth': 6,
        'aliases': ['SBI', 'STATE BANK'],
        'levels': ['ho','lho','zo','rbo','branch'],
        'type_map': {
            'HO':'ho','LHO':'lho','Local Head Office':'lho',
            'SBI LHO':'lho','ZONE':'zo',
            'ZO':'zo','RBO':'rbo','Branch':'branch',
        },
    },
    # ── UCO Bank ──────────────────────────────────────────────────────────────
    # Tenant → Customer → HO → ZO → Branch
    'UCO BANK': {
        'depth': 4,
        'levels': ['ho','zo','branch'],
        'type_map': {'HO':'ho','ZO':'zo','Branch':'branch'},
    },
    # ── Union Bank of India ───────────────────────────────────────────────────
    # Tenant → Customer → Central Office → ZO → RO → Branch
    'UNION BANK OF INDIA': {
        'depth': 5,
        'levels': ['ho','zo','ro','branch'],
        'type_map': {
            'Central Office':'ho','ZO':'zo','RO':'ro','Branch':'branch',
        },
    },
}

# ── GENERIC fallback type map (covers all known TB asset types from v8 output)
GENERIC_LEVEL_MAP = {
    # HO / top
    'Head Office BOI':'ho','Head Office BOB':'ho','Head Office CB':'ho',
    'Bank-Head Office':'ho','Corporate Office':'ho','Central Office':'ho',
    'Demo HO':'ho','Head Office':'ho','HO':'ho',
    # NBG / FGMO
    'NBG BOI':'nbg','Demo NBG':'nbg','NBG':'nbg','FGMO':'nbg',
    # LHO
    'LHO':'lho','Local Head Office':'lho',
    # ZO
    'Zonal Office BOI':'zo','Zonal Office BOB':'zo','Bank-Zonal Office':'zo',
    'Demo ZO':'zo','ZO':'zo','Zonal Office':'zo','zo':'zo',
    # RO
    'Regional Office BOB':'ro','Regional Office CB':'ro',
    'RO':'ro','Regional Office':'ro','ro':'ro',
    # CO / Circle Office
    'CO':'co','Circle Office':'co','Circle':'co',
    # RBO
    'RBO':'rbo','Regional Banking Office':'rbo',
    # Branch
    'Branch BOI':'branch','Branch BOB':'branch','Branch CB':'branch',
    'Bank-Branch':'branch','Demo Branch':'branch',
    'Branch':'branch','branch':'branch','Site':'branch','Location':'branch',
    # Special TB asset types discovered from v8 (these are EVENT flags stored as assets — ignore)
    'POWER OFF':'_ignore','MAINS ON':'_ignore','NETWORK':'_ignore',
    'DVR/NVR OFF':'_ignore','BATTERY LOW':'_ignore','SYSTEM ON':'_ignore',
    'FIRE ALARM SYSTEM OFF':'_ignore','FIRE ALARM SYSTEM ON':'_ignore',
    'INTRUSION ALARM SYSTEM OFF':'_ignore','INTRUSION ALARM SYSTEM ON':'_ignore',
    'TIME LOCK SYSTEM ON':'_ignore','HDD ERROR RESTORED':'_ignore',
    'CAMERA CONNECTION ESTABLISHED':'_ignore','CAMERA TAMPERED RESTORED CH 1':'_ignore',
    'CAMERA TAMPERED RESTORED CH 8':'_ignore',
    'INTEGRATED ALARM SYSTEM ACTIVATION RESTORED':'_ignore',
    'INTEGRATED ALARM SYSTEM ACTIVATION RESTORED':'_ignore',
    'INTRUSION ALARM SYSTEM FAULT':'_ignore',
    'INTRUSION ALARM SYSTEM ACTIVATION RESTORED':'_ignore',
    'FIRE ALARM SYSTEM ACTIVATION RESTORED':'_ignore',
    'FIRE ALARM SYSTEM ACTIVATION RESTORED':'_ignore',
    'CAMERA CONNECTION ESTABLISHED CH 2':'_ignore',
    'Restricted':'_ignore','Unloading':'_ignore','Loading':'_ignore',
    'Mine site':'_ignore','default':'_ignore',
    # Bank-specific office types
    'Regional Office BOI':'ro','Bank-Regional Office':'ro',
}

def get_asset_level(asset_type, bank_name=''):
    """Returns canonical level for an asset type, using per-bank map first."""
    # Check per-bank exact map
    for bank_key, bank_cfg in BANK_HIERARCHY.items():
        if bank_key.lower() in (bank_name or '').lower():
            if asset_type in bank_cfg['type_map']:
                return bank_cfg['type_map'][asset_type]
    # Generic map
    if asset_type in GENERIC_LEVEL_MAP:
        return GENERIC_LEVEL_MAP[asset_type]
    # Substring match
    alow = asset_type.lower()
    for k, v in GENERIC_LEVEL_MAP.items():
        if k.lower() in alow and v != '_ignore':
            return v
    return 'other'


import re
_LEVEL_KW = [
    (re.compile(r'\bLHO\b|Local\s+Head\s+Office', re.I), 'lho'),
    (re.compile(r'\bRBO\b|Regional\s+Banking', re.I),     'rbo'),
    (re.compile(r'\bNBG\b|\bFGMO\b', re.I),              'nbg'),
    (re.compile(r'\bZO\b|\bZONE\b|Zonal\s+Office', re.I), 'zo'),
    (re.compile(r'\bRO\b|Regional\s+Office', re.I),       'ro'),
    (re.compile(r'\bCO\b|Circle\s+Office', re.I),         'co'),
    (re.compile(r'\bHO\b|Head\s+Office|Corporate\s+Office|Central\s+Office', re.I), 'ho'),
]
def classify_entity_level(name):
    """Detect hierarchy level from any entity name (customer or asset title)."""
    for pat, level in _LEVEL_KW:
        if pat.search(name or ''):
            return level
    return None

# ── CLIENT ATTRIBUTE KEYS (from spec + v8 discovery) ─────────────────────────
CLIENT_KEYS = [
    'dexter_config','Hikvision_NVR_cameraInfo','Hikvision_NVR_deviceID',
    'Hikvision_NVR_deviceName','Hikvision_NVR_deviceType',
    'Hikvision_NVR_firmwareVersion','Hikvision_NVR_hardwareVersion',
    'Hikvision_NVR_HDDInfo','Hikvision_NVR_macAddress',
    'Hikvision_NVR_Manufacturer','Hikvision_NVR_model',
    'Hikvision_NVR_Processor','Hikvision_NVR_serialNumber',
    'lastUpdate','unknown',
]

# ── SERVER ATTRIBUTE KEYS (spec + v8 new keys) ───────────────────────────────
SERVER_KEYS = [
    'accessControl','accessControlDoor','accessControlHealth','accessControlStatus',
    'acsDoorOpen_history','acsOff_history','acsTamper_history',
    'active','alarm','alarmFlag',
    'bas','basAlarmCreatedTime','basFault_history','basHealth','basOff_history',
    'basStatus','basSystem',
    'BATTERY LOW','BATTERY REVERSE','BATTERY ON',  # ← new from v8
    'branch_id','branchName',
    'CAMERA CONNECTION ESTABLISHED','CAMERA DISCONNECT',
    'CAMERA TAMPER','CAMERA TAMPERED RESTORED',
    'cameraDisconnectCH1_history','cameraDisconnectCH2_history',
    'cameraDisconnectCH3_history','cameraDisconnectCH4_history',
    'cameraDisconnectCH5_history','cameraDisconnectCH6_history',
    'cameraDisconnectCH7_history','cameraDisconnectCH8_history',
    'cameraDisconnectCH9_history','cameraDisconnectCH10_history',
    'cameraDisconnectCH11_history','cameraDisconnectCH12_history',
    'cameraDisconnectCH13_history','cameraDisconnectCH14_history',
    'cameraDisconnectCH15_history','cameraDisconnectCH16_history',
    'cameraDisconnectCount','cameraLinkStatus','cameraStatus',
    'cameraTamperCH1_history','cameraTamperCH2_history',
    'cameraTamperCH3_history','cameraTamperCH4_history',
    'cameraTamperCH5_history','cameraTamperCH6_history',
    'cameraTamperCH7_history','cameraTamperCH8_history',
    'cameraTamperCH9_history','cameraTamperCH10_history',
    'cameraTamperCH11_history','cameraTamperCH12_history',
    'cameraTamperCH13_history','cameraTamperCH14_history',
    'cameraTamperCH15_history','cameraTamperCH16_history',
    'cameraTamperCount','care','cctv','cctvAlarmCreatedTime','cctvStatus',
    'count_CH','count_HDD','critical','deviceName','deviceType',
    'DVR/NVR OFF','DVR/NVR ON','dvrNvrOff_history','eventMetadata',
    'fas','fasAlarmCreatedTime','fasf','fasFault_history','fasHealth',
    'fasOff_history','fasStatus','fasSystem',
    'Faulty Device(Intrusion)','Faulty Device(Time Lock)',
    'FIRE ALARM SYSTEM ACTIVATE','FIRE ALARM SYSTEM ACTIVATION RESTORED',
    'FIRE ALARM SYSTEM ACTIVE','FIRE ALARM SYSTEM FAULT',
    'FIRE ALARM SYSTEM FAULT CONDITION RESTORED',
    'FIRE ALARM SYSTEM OFF','FIRE ALARM SYSTEM ON',
    'fireAlarmStatus','fireAlarmType','formattedBranchName',
    'gateway','gatewayAlarmCreatedTime','gatewayStatus','gatewayType',
    'gwHealth','gwStatus',
    'HDD ERROR','HDD ERROR RESTORED','hddandDvrNvr','hddError_history','hddStatus',
    'Healthy Device(Intrusion)','Healthy Device(Time Lock)',
    'ias','iasAlarmCreatedTime','iasf','iasFault_history','iasHealth',  # ← iasAlarmCreatedTime new
    'iasOff_history','iasStatus','iasSystem','imei_id',
    'Inactive Device(Intrusion)','Inactive Device(Time Lock)',
    'inactiveDeviceName','inactiveReason','inactiveSince','inactivityAlarmTime',
    # Integrated Alarm System (new from v8)
    'INTEGRATED ALARM SYSTEM OFF','INTEGRATED ALARM SYSTEM ON',
    'INTEGRATED ALARM SYSTEM FAULT CONDITION RESTORED',
    'INTEGRATED ALARM SYSTEM ACTIVATION RESTORED',
    'INTEGRATED ALARM SYSTEM ACTIVE',
    'INTRUSION ALARM FAULT CONDITION RESTORED',
    'INTRUSION ALARM SYSTEM ACTIVATE','INTRUSION ALARM SYSTEM ACTIVATION RESTORED',
    'INTRUSION ALARM SYSTEM ACTIVE','INTRUSION ALARM SYSTEM FAULT',
    'INTRUSION ALARM SYSTEM OFF','INTRUSION ALARM SYSTEM ON',
    'intrusionStatus','intrusionType',
    'lastActivityTime','lastConnectTime','lastDisconnectTime','lastUpdate',
    'lowDurationCameras','MAINS ON','major','nbgName',
    'NETWORK','notification','nvrStatus','nvrType','org_id','POWER OFF',
    'provisionState','severity','status','subsystems','SYSTEM ON','systemHealth',
    'TIME LOCK DOOR CLOSE','TIME LOCK DOOR OPEN',
    'TIME LOCK SYSTEM OFF','TIME LOCK SYSTEM ON',
    'TIME LOCK SYSTEM TAMPER','TIME LOCK TAMPER RESTORED',
    'timeLock','timeLockAlarmCreatedTime','timeLockDoor','timeLockHealth',
    'timeLockMiliTime','timeLockStatus',
    'tlsDoorOpen_history','tlsOff_history','tlsTamper_history',
    'tlStatus','tlType',
    'Total System(Intrusion)','Total System(Time Lock)',
    'ts','type','undefined','unknown','unknown_status',
    'usage_history',
    # New from v8 key discovery
    'usage_daily','usage_last_7_days','usage_last_15_days',
    'customer_title',
    'warning','zoName','zone_name',
]

# ── TELEMETRY KEYS ────────────────────────────────────────────────────────────
TELEMETRY_KEYS = [
    'target_sw_tag','target_sw_title','target_sw_ts','target_sw_version',
    'sw_state','sw_version','fw_version','fw_state',
    'cavlidata_ontime','Total_Data_Usage',
    'sim_iccid','sim_operator','signal_strength','network_type','ip_address',
    'arrLat','arrLon','latitude','longitude',
    'cpu','ram','disk','temperature','uptime','battery_voltage',
    'memUsage','cpuUsage',
    'BAS_Downtime_Minutes','NVR_Downtime_Minutes','FAS_Downtime_Minutes',
    'IAS_Downtime_Minutes','ACS_Downtime_Minutes',
    'cameraCount','cameraOnline','cameraOffline',
    'nvrStatus','hddStatus','hddCapacity','hddUsed','recordingStatus',
    'gwStatus','powerStatus','upsStatus',
    'fasStatus','iasStatus','basStatus','accessControlStatus','timeLockStatus',
    'usage_history','lastUpdate','inactiveSince','inactiveReason',
]

# Auth
session        = requests.Session()
session.verify = False
resp = session.post(f'{TB_HOST}/api/auth/login',
    json={'username':TB_EMAIL,'password':TB_PASSWORD},
    headers={'Content-Type':'application/json'}, timeout=15)
if resp.status_code != 200:
    raise Exception(f'Login failed HTTP {resp.status_code}: {resp.text}')
JWT_TOKEN    = resp.json()['token']
AUTH_HEADERS = {'X-Authorization':f'Bearer {JWT_TOKEN}',
                'Content-Type':'application/json'}
print(f'✅ Authenticated — {TB_HOST}')
print(f'   CLIENT keys  : {len(CLIENT_KEYS)}')
print(f'   SERVER keys  : {len(SERVER_KEYS)}')
print(f'   TELEMETRY    : {len(TELEMETRY_KEYS)}')
print(f'   Bank configs : {len(BANK_HIERARCHY)}')


✅ Authenticated — https://seple.iot-private.cloud
   CLIENT keys  : 15
   SERVER keys  : 189
   TELEMETRY    : 52
   Bank configs : 12


---
## Cell 4 — Core Helpers

In [18]:
def safe_float(v, d=0.0):
    try:   return float(v or 0)
    except: return d

def safe_int(v, d=0):
    try:   return int(float(v or 0))
    except: return d

def to_json(v):
    if isinstance(v,(dict,list)): return v
    if isinstance(v,str):
        try: return json.loads(v)
        except: return None
    return None

def epoch_ms(ts):
    try:
        if ts and float(ts) > 0:
            return datetime.utcfromtimestamp(float(ts)/1000).strftime('%Y-%m-%d %H:%M')
    except: pass
    return ''

def is_fault(v):
    if v is None: return False
    return str(v).strip().upper() in (
        'OFFLINE','OFF','FAULT','ERROR','INACTIVE',
        'DISCONNECTED','DOWN','FAILED','0','FALSE','N/A','FAILED_UPDATE')

def is_active(v):
    return v not in (None,False,'false','False',0,'0','','null')

def first(*vals):
    for v in vals:
        if v not in (None,'','null','None'): return v
    return ''

# ── Paginated GET ─────────────────────────────────────────────────────────────
def paginate(url_tpl, page_size=100):
    items, page = [], 0
    while True:
        url  = url_tpl.format(page=page, size=page_size)
        resp = session.get(url, headers=AUTH_HEADERS, timeout=30)
        if resp.status_code == 401: raise Exception('JWT expired — re-run Cell 3')
        if resp.status_code != 200:
            print(f'  ⚠️ HTTP {resp.status_code}: {url[:80]}')
            break
        data = resp.json()
        items.extend(data.get('data',[]))
        if not data.get('hasNext',False): break
        page += 1
        time.sleep(REQUEST_DELAY)
    return items

# ── Attribute fetch (chunked by key list) ─────────────────────────────────────
def fetch_attr_scope_keys(etype, eid, scope, keys):
    result = {}
    for i in range(0, len(keys), 50):
        chunk = keys[i:i+50]
        url   = (f'{TB_HOST}/api/plugins/telemetry/{etype}/{eid}'
                 f'/values/attributes/{scope}?keys={",".join(chunk)}')
        try:
            r = session.get(url, headers=AUTH_HEADERS, timeout=20)
            if r.status_code == 200:
                for item in r.json():
                    result[item['key']] = item['value']
        except: pass
        time.sleep(0.02)
    return result

def fetch_attr_scope_all(etype, eid, scope):
    url = f'{TB_HOST}/api/plugins/telemetry/{etype}/{eid}/values/attributes/{scope}'
    try:
        r = session.get(url, headers=AUTH_HEADERS, timeout=15)
        if r.status_code == 200:
            return {item['key']: item['value'] for item in r.json()}
    except: pass
    return {}

def fetch_all_attributes(etype, eid):
    """Fetch explicit keys + full fallback scan across all 3 scopes."""
    attrs = {}
    # Explicit key fetch (CLIENT + SERVER)
    attrs.update(fetch_attr_scope_keys(etype, eid, 'CLIENT_SCOPE', CLIENT_KEYS))
    attrs.update(fetch_attr_scope_keys(etype, eid, 'SERVER_SCOPE', SERVER_KEYS))
    # Full fallback — catches any key not in our lists
    for scope in ['SERVER_SCOPE','CLIENT_SCOPE','SHARED_SCOPE']:
        for k, v in fetch_attr_scope_all(etype, eid, scope).items():
            if k not in attrs:
                attrs[k] = v
    return attrs

# ── Telemetry fetch (chunked) ─────────────────────────────────────────────────
def fetch_telemetry(device_id, keys):
    result = {}
    for i in range(0, len(keys), 50):
        chunk = keys[i:i+50]
        url   = (f'{TB_HOST}/api/plugins/telemetry/DEVICE/{device_id}'
                 f'/values/timeseries?keys={",".join(chunk)}&useStrictDataTypes=false')
        try:
            r = session.get(url, headers=AUTH_HEADERS, timeout=20)
            if r.status_code == 200:
                for k, entries in r.json().items():
                    if entries:
                        result[f'tele_{k}']    = entries[0].get('value')
                        result[f'tele_{k}_ts'] = epoch_ms(entries[0].get('ts'))
        except: pass
        time.sleep(0.02)
    return result

# ── Relations ─────────────────────────────────────────────────────────────────
_rel_cache = {}
def get_parents(etype, eid):
    key = (etype, eid)
    if key in _rel_cache: return _rel_cache[key]
    url = f'{TB_HOST}/api/relations?toId={eid}&toType={etype}'
    try:
        r = session.get(url, headers=AUTH_HEADERS, timeout=15)
        if r.status_code == 200:
            parents = [{'entity_type': rel.get('from',{}).get('entityType',''),
                        'entity_id':   rel.get('from',{}).get('id','')}
                       for rel in (r.json() if isinstance(r.json(), list) else [])]
            _rel_cache[key] = parents
            time.sleep(REQUEST_DELAY)
            return parents
    except: pass
    _rel_cache[key] = []
    return []

print('✅ Core helpers ready.')


✅ Core helpers ready.


---
## Cell 5 — Fetch Customers (Banks = Top-Level NBGs)

In [19]:
from tqdm import tqdm

print('📡 Fetching Customers (Banks) ...')
raw_custs = paginate(
    TB_HOST + '/api/customers?pageSize={size}&page={page}&sortProperty=title&sortOrder=ASC'
)
print(f'   Found {len(raw_custs)} customers.\n')

customers = []
for c in tqdm(raw_custs, desc='Customers'):
    cid   = c.get('id',{}).get('id','')
    title = c.get('title','')
    attrs = fetch_attr_scope_all('CUSTOMER', cid, 'SERVER_SCOPE')
    attrs.update(fetch_attr_scope_all('CUSTOMER', cid, 'CLIENT_SCOPE'))

    # Match to known bank config
    bank_cfg = None
    for bank_name, cfg in BANK_HIERARCHY.items():
        aliases = cfg.get('aliases', [])
        tlow = title.lower()
        if (bank_name.lower() in tlow
            or tlow in bank_name.lower()
            or any(a.lower() in tlow for a in aliases)):
            bank_cfg = cfg
            bank_cfg['bank_name'] = bank_name
            break

    customers.append({
        'customer_id':    cid,
        'customer_title': title,
        'bank_name':      bank_cfg['bank_name'] if bank_cfg else title,
        'bank_depth':     bank_cfg['depth']     if bank_cfg else 3,
        'nbg_name':       first(attrs.get('nbgName'), attrs.get('nbg_name'), title),
        'region':         first(attrs.get('region'),  attrs.get('circle'), ''),
        'state':          first(attrs.get('state'),   c.get('state',''),   ''),
        'city':           first(attrs.get('city'),    c.get('city',''),    ''),
        'address':        first(attrs.get('address'), c.get('address',''),''),
        'email':          first(attrs.get('email'),   c.get('email',''),   ''),
        'phone':          first(attrs.get('phone'),   c.get('phone',''),   ''),
        'created_time':   epoch_ms(c.get('createdTime')),
        '_attrs':         attrs,
    })
    time.sleep(REQUEST_DELAY)

cust_map   = {c['customer_id']: c for c in customers}
cust_title = {c['customer_id']: c['customer_title'] for c in customers}
cust_bank  = {c['customer_id']: c['bank_name'] for c in customers}

print(f'\n✅ {len(customers)} customers. Banks matched:')
for c in customers:
    print(f'  {c["customer_title"]:<40}  depth={c["bank_depth"]}  bank={c["bank_name"]}')


📡 Fetching Customers (Banks) ...
   Found 189 customers.



Customers: 100%|██████████| 189/189 [00:29<00:00,  6.40it/s]


✅ 189 customers. Banks matched:
  BAGAHA CC                                 depth=3  bank=BAGAHA CC
  Bank of Baroda demo                       depth=5  bank=BANK OF BARODA
  BAS                                       depth=3  bank=BAS
  BHAWAL                                    depth=3  bank=BHAWAL
  BR ABC                                    depth=3  bank=BR ABC
  BRANCH ADITYAPUR                          depth=3  bank=BRANCH ADITYAPUR
  BRANCH AGARTALA                           depth=3  bank=BRANCH AGARTALA
  BRANCH AIZWAL                             depth=3  bank=BRANCH AIZWAL
  BRANCH ALIPURDUAR                         depth=3  bank=BRANCH ALIPURDUAR
  BRANCH AMALNER                            depth=3  bank=BRANCH AMALNER
  BRANCH_AMTALA                             depth=3  bank=BRANCH_AMTALA
  BRANCH_APC ROAD                           depth=3  bank=BRANCH_APC ROAD
  BRANCH ARAMBAGH                           depth=3  bank=BRANCH ARAMBAGH
  BRANCH ASHANGI                            

---
## Cell 6 — Fetch ALL Assets + Classify by Bank-Specific Level

**Key insight from v8**: Many asset `type` values in TB are actually EVENT flag names
(e.g. `POWER OFF`, `FIRE ALARM SYSTEM OFF`). These are ignored via `_ignore` level.

In [20]:
print('📡 Fetching ALL Assets ...')
raw_assets = paginate(
    TB_HOST + '/api/tenant/assets?pageSize={size}&page={page}&sortProperty=name&sortOrder=ASC'
)
print(f'   Found {len(raw_assets)} assets. Fetching attributes + classifying ...\n')

assets = []
for a in tqdm(raw_assets, desc='Assets'):
    aid   = a.get('id',{}).get('id','')
    atype = a.get('type','')
    cid   = (a.get('customerId') or {}).get('id','')
    bank  = cust_bank.get(cid, '')
    level = get_asset_level(atype, bank)

    if level == '_ignore':
        continue   # skip event-flag pseudo-assets

    attrs = {}
    for scope in ['SERVER_SCOPE','CLIENT_SCOPE']:
        attrs.update(fetch_attr_scope_all('ASSET', aid, scope))

    assets.append({
        'asset_id':    aid,
        'asset_name':  a.get('name',''),
        'asset_type':  atype,
        'level':       level,
        'customer_id': cid,
        'bank_name':   bank,
        'created':     epoch_ms(a.get('createdTime')),
        # Common hierarchy labels stored in asset attributes
        'nbg_name':    first(attrs.get('nbgName'),    attrs.get('nbg_name'),   ''),
        'zo_name':     first(attrs.get('zoName'),     attrs.get('zo_name'),
                             attrs.get('zoneName'),   ''),
        'zo_code':     first(attrs.get('zoCode'),     attrs.get('zone_code'),  ''),
        'ro_name':     first(attrs.get('roName'),     attrs.get('ro_name'),    ''),
        'branch_name': first(attrs.get('branchName'), attrs.get('branch_name'),''),
        'branch_code': first(attrs.get('branchCode'), attrs.get('branch_code'),''),
        'display_name':first(attrs.get('displayName'), a.get('name',''),       ''),
        'address':     first(attrs.get('address'),    ''),
        'city':        first(attrs.get('city'),       ''),
        'state':       first(attrs.get('state'),      ''),
        'pincode':     first(attrs.get('pincode'),    attrs.get('zip',''),     ''),
        'latitude':    first(attrs.get('latitude'),   attrs.get('arrLat'),     ''),
        'longitude':   first(attrs.get('longitude'),  attrs.get('arrLon'),     ''),
        'install_date':first(attrs.get('installationDate'), ''),
        'go_live_date':first(attrs.get('goLiveDate'),       ''),
        'contract':    first(attrs.get('contractType'),     ''),
        'sla':         first(attrs.get('slaTier'),          ''),
        '_attrs':      attrs,
    })
    time.sleep(REQUEST_DELAY)

asset_map = {a['asset_id']: a for a in assets}

# Summary
level_dist = Counter(a['level'] for a in assets)
print(f'\n✅ {len(assets)} assets classified (ignored event-flag pseudo-assets).')
print('\nLevel distribution:')
for lvl, cnt in sorted(level_dist.items()):
    print(f'   {lvl:<15} {cnt}')

print('\nPer-bank asset breakdown:')
bank_asset_dist = defaultdict(Counter)
for a in assets:
    bank_asset_dist[a['bank_name']][a['level']] += 1
for bank, lc in sorted(bank_asset_dist.items()):
    lstr = '  '.join(f'{l}:{n}' for l,n in sorted(lc.items()))
    print(f'   {bank:<35} {lstr}')


📡 Fetching ALL Assets ...
   Found 160 assets. Fetching attributes + classifying ...



Assets: 100%|██████████| 160/160 [00:24<00:00,  6.64it/s]


✅ 157 assets classified (ignored event-flag pseudo-assets).

Level distribution:
   branch          120
   ho              5
   nbg             6
   ro              8
   zo              18

Per-bank asset breakdown:
                                       branch:4  ho:1  zo:6
   BANK OF BARODA                      branch:12  ho:1  ro:3  zo:1
   BANK OF INDIA                       branch:98  ho:1  nbg:5  zo:10
   CANARA BANK                         branch:5  ho:1  ro:5
   New Structure                       branch:1  ho:1  nbg:1  zo:1


---
## Cell 7 — Fetch ALL Devices + Attributes + Telemetry

3 API calls per device: CLIENT attrs, SERVER attrs, TELEMETRY.  
⏳ ~6–10 min for 1000+ devices.

In [21]:
print('📡 Fetching ALL Devices ...')
all_devices = paginate(
    TB_HOST + '/api/tenant/devices?pageSize={size}&page={page}&sortProperty=name&sortOrder=ASC'
)
print(f'   Found {len(all_devices)} devices. Fetching all data ...\n')

device_data, fetch_errors = [], []

for device in tqdm(all_devices, desc='Devices', unit='dev'):
    dev_id   = device.get('id',{}).get('id','')
    dev_name = device.get('name','UNKNOWN')
    cust_id  = (device.get('customerId') or {}).get('id','')
    bank     = cust_bank.get(cust_id, '')
    try:
        # ── Fetch all attributes (explicit + full fallback) ────────────────
        attrs = fetch_all_attributes('DEVICE', dev_id)

        # ── Fetch all telemetry ────────────────────────────────────────────
        tele  = fetch_telemetry(dev_id, TELEMETRY_KEYS)

        # Merge: attrs win over telem on conflicts
        merged = {**tele, **attrs}

        # Device metadata
        merged['_device_id']      = dev_id
        merged['_device_name']    = dev_name
        merged['_device_type']    = device.get('type','')
        merged['_device_profile'] = device.get('deviceProfileName','')
        merged['_customer_id']    = cust_id
        merged['_customer_name']  = cust_title.get(cust_id,'')
        merged['_bank_name']      = bank
        merged['_created_time']   = device.get('createdTime','')

        device_data.append(merged)
    except Exception as e:
        fetch_errors.append({'id':dev_id,'name':dev_name,'error':str(e)})
    time.sleep(REQUEST_DELAY)

# Build quick index
dev_index = {d['_device_id']: d for d in device_data}

print(f'\n✅ Fetched  : {len(device_data)} devices')
print(f'   Errors   : {len(fetch_errors)}')
if fetch_errors:
    for e in fetch_errors[:5]: print(f'   ⚠️  {e["name"]}: {e["error"]}')
all_keys = set(k for d in device_data for k in d)
print(f'\n📊 Unique keys across all devices: {len(all_keys)}')


📡 Fetching ALL Devices ...
   Found 184 devices. Fetching all data ...



Devices: 100%|██████████| 184/184 [02:20<00:00,  1.31dev/s]


✅ Fetched  : 184 devices
   Errors   : 0

📊 Unique keys across all devices: 598


---
## Cell 8 — Adaptive Hierarchy Resolver (Bank-Aware)

For each device:
1. Read embedded attributes (`nbgName`, `zoName`, `branchName`) → fastest
2. Walk relation chain upward, classifying each asset by its bank-specific level
3. Fallback to customer_map for bank/NBG name

Produces a **full_path** column: `Bank → HO → NBG → ZO → Branch`

In [22]:
LEVEL_ORDER = ['ho','nbg','lho','zo','ro','co','rbo','sub_branch','branch']

def resolve_hierarchy_walk(etype, eid, bank_name='', depth=0):
    """
    Recursively walk parent relations.
    Returns dict with any levels found: ho_name, nbg_name, zo_name, etc.
    """
    if depth >= MAX_RELATION_DEPTH:
        return {}
    result = {}
    for p in get_parents(etype, eid):
        ptype = p['entity_type']
        pid   = p['entity_id']
        if ptype == 'CUSTOMER' and pid in cust_map:
            c = cust_map[pid]
            result.setdefault('bank_name',  c['bank_name'])
            result.setdefault('nbg_name',   c['nbg_name'])
            # ── NEW: classify customer entity as hierarchy level ──────
            clvl = classify_entity_level(c['customer_title'])
            cname = c['customer_title']
            if   clvl == 'lho': result.setdefault('lho_name', cname)
            elif clvl == 'rbo': result.setdefault('rbo_name', cname)
            elif clvl == 'zo':  result.setdefault('zo_name',  cname)
            elif clvl == 'ro':  result.setdefault('ro_name',  cname)
            elif clvl == 'co':  result.setdefault('co_name',  cname)
            elif clvl == 'ho':  result.setdefault('ho_name',  cname)
            elif clvl == 'nbg': result.setdefault('nbg_name', cname)
        elif ptype == 'ASSET' and pid in asset_map:
            a     = asset_map[pid]
            level = a['level']
            if level == '_ignore': continue
            name  = a['display_name'] or a['asset_name']
            if level == 'ho':
                result.setdefault('ho_name',   name)
                result.setdefault('ho_id',     pid)
            elif level == 'nbg':
                result.setdefault('nbg_name',  first(a['nbg_name'], name))
                result.setdefault('nbg_id',    pid)
            elif level == 'lho':
                result.setdefault('lho_name',  name)
                result.setdefault('lho_id',    pid)
            elif level == 'zo':
                result.setdefault('zo_name',   first(a['zo_name'], name))
                result.setdefault('zo_code',   a['zo_code'])
                result.setdefault('zo_id',     pid)
                result.setdefault('zo_state',  a['state'])
            elif level == 'ro':
                result.setdefault('ro_name',   first(a['ro_name'], name))
                result.setdefault('ro_id',     pid)
            elif level == 'co':
                result.setdefault('co_name',   name)
                result.setdefault('co_id',     pid)
            elif level == 'rbo':
                result.setdefault('rbo_name',  name)
                result.setdefault('rbo_id',    pid)
            elif level in ('branch','sub_branch'):
                result.setdefault('branch_name',    first(a['branch_name'], name))
                result.setdefault('branch_code',    a['branch_code'])
                result.setdefault('branch_address', a['address'])
                result.setdefault('branch_city',    a['city'])
                result.setdefault('branch_state',   a['state'])
                result.setdefault('branch_pincode', a['pincode'])
                result.setdefault('branch_lat',     a['latitude'])
                result.setdefault('branch_lon',     a['longitude'])
                result.setdefault('install_date',   a['install_date'])
                result.setdefault('go_live_date',   a['go_live_date'])
                result.setdefault('contract',       a['contract'])
                result.setdefault('sla',            a['sla'])
            # Always walk higher
            upper = resolve_hierarchy_walk('ASSET', pid, bank_name, depth+1)
            for k, v in upper.items():
                result.setdefault(k, v)
    return result


def build_hierarchy(attrs):
    """
    3-step hierarchy resolution for a device's merged attrs dict.
    Returns a complete hierarchy dict with all possible levels.
    """
    h = {
        # Step A: embedded attribute values (fastest, most reliable)
        'bank_name':     attrs.get('_bank_name', ''),
        'nbg_name':      first(attrs.get('nbgName'),    attrs.get('nbg_name'),   ''),
        'zo_name':       first(attrs.get('zoName'),     attrs.get('zo_name'),
                               attrs.get('zoneName'),   ''),
        'zo_code':       first(attrs.get('zoCode'),     attrs.get('zone_code'),  ''),
        'branch_name':   first(attrs.get('branchName'), attrs.get('branch_name'),
                               attrs.get('formattedBranchName'), ''),
        'branch_code':   first(attrs.get('branch_id'),  attrs.get('branchCode'), ''),
        'branch_address':first(attrs.get('address'),    attrs.get('addr'),        ''),
        'branch_city':   first(attrs.get('city'),       ''),
        'branch_state':  first(attrs.get('state'),      ''),
        'branch_lat':    first(attrs.get('arrLat'),     attrs.get('latitude'),    ''),
        'branch_lon':    first(attrs.get('arrLon'),     attrs.get('longitude'),   ''),
        # Extra levels (may come from relation walk)
        'ho_name':'','lho_name':'','ro_name':'','co_name':'','rbo_name':'',
    }

    # Step B: relation walk to fill any gaps
    if not h['nbg_name'] or not h['branch_name']:
        rel = resolve_hierarchy_walk(
            'DEVICE', attrs['_device_id'], attrs.get('_bank_name','')
        )
        for k, v in rel.items():
            if not h.get(k) and v:
                h[k] = v

    # Step C: customer fallback for bank/NBG name
    cid = attrs.get('_customer_id','')
    if not h['nbg_name'] and cid in cust_map:
        c = cust_map[cid]
        h['nbg_name']  = c['nbg_name']
        h['bank_name'] = c['bank_name']

    # Build full_path string (bank-aware)
    path_parts = []
    for field in ['bank_name','ho_name','nbg_name','lho_name',
                  'zo_name','ro_name','co_name','rbo_name','branch_name']:
        v = h.get(field,'')
        if v: path_parts.append(v)
    h['full_path'] = ' → '.join(path_parts)
    h['hierarchy_depth'] = len(path_parts)

    return h

# Run hierarchy resolution for all devices
print('🔗 Resolving hierarchy for all devices ...')
for d in tqdm(device_data, desc='Hierarchy', unit='dev'):
    d['_hierarchy'] = build_hierarchy(d)

total = len(device_data)
def pct(n): return f'{n}/{total} ({100*n//max(total,1)}%)'
has_nbg    = sum(1 for d in device_data if d['_hierarchy'].get('nbg_name'))
has_zo     = sum(1 for d in device_data if d['_hierarchy'].get('zo_name'))
has_branch = sum(1 for d in device_data if d['_hierarchy'].get('branch_name'))
print(f'\n✅ Hierarchy resolved:')
print(f'   Bank/NBG : {pct(has_nbg)}')
print(f'   Zone     : {pct(has_zo)}')
print(f'   Branch   : {pct(has_branch)}')

# Sample full_path output
print('\nSample full paths:')
for d in device_data[:8]:
    print(f'  {d["_hierarchy"]["full_path"] or "(unresolved)"}')


🔗 Resolving hierarchy for all devices ...


Hierarchy: 100%|██████████| 184/184 [00:05<00:00, 30.94dev/s] 


✅ Hierarchy resolved:
   Bank/NBG : 134/184 (72%)
   Zone     : 129/184 (70%)
   Branch   : 121/184 (65%)

Sample full paths:
  BANK OF BARODA → NBG EAST → ZO HOWRAH → ABC
  NBG EAST → ZO HOWRAH → Branch TR
  TLS CUS DEMO → TLS CUS DEMO → ZO Muzaffapur
  BANK OF BARODA → ZO(Kolkata) → ZO KOLKATA → BRANCH_NSB AIRPORT
  BANK OF BARODA → Bank of Baroda demo → ZO KOLKATA → BRANCH AMTALA
  BANK OF BARODA → ZO(Kolkata) → RO(KMR) → BRANCH_APC ROAD
  BANK OF BARODA → ZO(Kolkata) → ZO_Kolkata → Branch_Shakuntal_Park
  BANK OF BARODA → Bank of Baroda demo → ZO KOLKATA → BRANCH BARUIPUR


---
## Cell 9 — Build Master DataFrame (All 160+ Columns)

In [23]:
import pandas as pd

EVENT_FLAG_MAP = {
    'ev_power_off':           'POWER OFF',
    'ev_dvr_off':             'DVR/NVR OFF',
    'ev_dvr_on':              'DVR/NVR ON',
    'ev_hdd_error':           'HDD ERROR',
    'ev_hdd_restored':        'HDD ERROR RESTORED',
    'ev_battery_low':         'BATTERY LOW',
    'ev_battery_reverse':     'BATTERY REVERSE',   # new v9
    'ev_battery_on':          'BATTERY ON',         # new v9
    'ev_mains_on':            'MAINS ON',
    'ev_system_on':           'SYSTEM ON',
    'ev_network':             'NETWORK',
    'ev_cam_disconnect':      'CAMERA DISCONNECT',
    'ev_cam_tamper':          'CAMERA TAMPER',
    'ev_cam_tamper_rst':      'CAMERA TAMPERED RESTORED',
    'ev_cam_connect':         'CAMERA CONNECTION ESTABLISHED',
    'ev_fas_off':             'FIRE ALARM SYSTEM OFF',
    'ev_fas_on':              'FIRE ALARM SYSTEM ON',
    'ev_fas_fault':           'FIRE ALARM SYSTEM FAULT',
    'ev_fas_fault_rst':       'FIRE ALARM SYSTEM FAULT CONDITION RESTORED',
    'ev_fas_active':          'FIRE ALARM SYSTEM ACTIVE',
    'ev_fas_activate':        'FIRE ALARM SYSTEM ACTIVATE',
    'ev_fas_activate_rst':    'FIRE ALARM SYSTEM ACTIVATION RESTORED',
    'ev_ias_off':             'INTRUSION ALARM SYSTEM OFF',
    'ev_ias_on':              'INTRUSION ALARM SYSTEM ON',
    'ev_ias_fault':           'INTRUSION ALARM SYSTEM FAULT',
    'ev_ias_fault_rst':       'INTRUSION ALARM FAULT CONDITION RESTORED',
    'ev_ias_active':          'INTRUSION ALARM SYSTEM ACTIVE',
    'ev_ias_activate':        'INTRUSION ALARM SYSTEM ACTIVATE',
    'ev_ias_activate_rst':    'INTRUSION ALARM SYSTEM ACTIVATION RESTORED',
    'ev_int_off':             'INTEGRATED ALARM SYSTEM OFF',   # new v9
    'ev_int_on':              'INTEGRATED ALARM SYSTEM ON',    # new v9
    'ev_int_fault_rst':       'INTEGRATED ALARM SYSTEM FAULT CONDITION RESTORED',
    'ev_int_act_rst':         'INTEGRATED ALARM SYSTEM ACTIVATION RESTORED',
    'ev_int_active':          'INTEGRATED ALARM SYSTEM ACTIVE',
    'ev_tls_off':             'TIME LOCK SYSTEM OFF',
    'ev_tls_on':              'TIME LOCK SYSTEM ON',
    'ev_tls_tamper':          'TIME LOCK SYSTEM TAMPER',
    'ev_tls_tamper_rst':      'TIME LOCK TAMPER RESTORED',
    'ev_tls_door_open':       'TIME LOCK DOOR OPEN',
    'ev_tls_door_close':      'TIME LOCK DOOR CLOSE',
}

print(f'⚙️  Building master DataFrame for {len(device_data)} devices ...')
rows = []
for attrs in device_data:
    h  = attrs['_hierarchy']
    sh = to_json(attrs.get('systemHealth')) or {}

    def tele(k, default=None):
        return attrs.get(f'tele_{k}', attrs.get(k, default))

    # Per-channel histories (CH1–16)
    ch_dc = {f'camDC_ch{i}': json.dumps(to_json(attrs.get(f'cameraDisconnectCH{i}_history')) or [])
             for i in range(1,17)}
    ch_tp = {f'camTP_ch{i}': json.dumps(to_json(attrs.get(f'cameraTamperCH{i}_history')) or [])
             for i in range(1,17)}

    row = {
        # ── HIERARCHY (full, all levels) ──────────────────────────────────────
        'bank_name':       h.get('bank_name',  attrs.get('_bank_name','')),
        'ho_name':         h.get('ho_name',    ''),
        'nbg_name':        h.get('nbg_name',   ''),
        'lho_name':        h.get('lho_name',   ''),
        'zo_name':         h.get('zo_name',    ''),
        'zo_code':         h.get('zo_code',    ''),
        'zo_state':        h.get('zo_state',   ''),
        'ro_name':         h.get('ro_name',    ''),
        'co_name':         h.get('co_name',    ''),
        'rbo_name':        h.get('rbo_name',   ''),
        'branch_name':     h.get('branch_name', attrs.get('_device_name','')),
        'branch_code':     h.get('branch_code',''),
        'branch_address':  h.get('branch_address',''),
        'branch_city':     h.get('branch_city',''),
        'branch_state':    h.get('branch_state',''),
        'branch_pincode':  h.get('branch_pincode',''),
        'branch_lat':      h.get('branch_lat', ''),
        'branch_lon':      h.get('branch_lon', ''),
        'install_date':    h.get('install_date',''),
        'go_live_date':    h.get('go_live_date',''),
        'contract_type':   h.get('contract',   ''),
        'sla_tier':        h.get('sla',        ''),
        'full_path':       h.get('full_path',  ''),
        'hierarchy_depth': h.get('hierarchy_depth', 0),

        # ── DEVICE IDENTITY ───────────────────────────────────────────────────
        'device_id':       attrs['_device_id'],
        'device_name':     attrs['_device_name'],
        'device_type':     attrs.get('_device_type',''),
        'device_profile':  attrs.get('_device_profile',''),
        'customer_id':     attrs['_customer_id'],
        'customer_name':   attrs.get('_customer_name',''),
        'device_created':  epoch_ms(attrs.get('_created_time')),
        'org_id':          first(attrs.get('org_id'),''),
        'imei_id':         first(attrs.get('imei_id'),''),
        'provisionState':  first(attrs.get('provisionState'),''),
        'active':          first(attrs.get('active'),''),
        'device_status':   first(attrs.get('status'),''),

        # ── NVR HARDWARE (Hikvision client attrs) ─────────────────────────────
        'nvr_model':       first(attrs.get('Hikvision_NVR_model'),     attrs.get('nvrType'),''),
        'nvr_serial':      first(attrs.get('Hikvision_NVR_serialNumber'),''),
        'nvr_firmware':    first(attrs.get('Hikvision_NVR_firmwareVersion'),''),
        'nvr_hardware':    first(attrs.get('Hikvision_NVR_hardwareVersion'),''),
        'nvr_mac':         first(attrs.get('Hikvision_NVR_macAddress'),''),
        'nvr_manufacturer':first(attrs.get('Hikvision_NVR_Manufacturer'),''),
        'nvr_processor':   first(attrs.get('Hikvision_NVR_Processor'),''),
        'nvr_device_id_hik':first(attrs.get('Hikvision_NVR_deviceID'),''),
        'nvr_hdd_info':    json.dumps(to_json(attrs.get('Hikvision_NVR_HDDInfo')) or ''),
        'nvr_cam_info':    json.dumps(to_json(attrs.get('Hikvision_NVR_cameraInfo',
                                      tele('Hikvision_NVR_cameraInfo'))) or []),

        # ── SUBSYSTEM STATUSES ────────────────────────────────────────────────
        'nvr_status':      first(attrs.get('nvrStatus'),     tele('nvrStatus'),''),
        'hdd_status':      first(attrs.get('hddStatus'),     tele('hddStatus'),''),
        'hdd_capacity':    safe_float(tele('hddCapacity',0)),
        'hdd_used':        safe_float(tele('hddUsed',0)),
        'fas_status':      first(attrs.get('fasStatus'),     tele('fasStatus'),''),
        'fas_health':      first(attrs.get('fasHealth'),''),
        'fas_system':      first(attrs.get('fasSystem'),''),
        'fire_alarm_status':first(attrs.get('fireAlarmStatus'),''),
        'fire_alarm_type': first(attrs.get('fireAlarmType'),''),
        'ias_status':      first(attrs.get('iasStatus'),     tele('iasStatus'),''),
        'ias_health':      first(attrs.get('iasHealth'),''),
        'ias_system':      first(attrs.get('iasSystem'),''),
        'intrusion_status':first(attrs.get('intrusionStatus'),''),
        'intrusion_type':  first(attrs.get('intrusionType'),''),
        'bas_status':      first(attrs.get('basStatus'),     tele('basStatus'),''),
        'bas_health':      first(attrs.get('basHealth'),''),
        'bas_system':      first(attrs.get('basSystem'),''),
        'acs_status':      first(attrs.get('accessControlStatus'), tele('accessControlStatus'),''),
        'acs_health':      first(attrs.get('accessControlHealth'),''),
        'acs_door':        first(attrs.get('accessControlDoor'),''),
        'tls_status':      first(attrs.get('timeLockStatus'), attrs.get('tlStatus'),
                                 tele('timeLockStatus'),''),
        'tls_health':      first(attrs.get('timeLockHealth'),''),
        'tls_door':        first(attrs.get('timeLockDoor'),''),
        'tls_type':        first(attrs.get('tlType'),''),
        'gw_status':       first(attrs.get('gwStatus'),      attrs.get('gatewayStatus'),
                                 tele('gwStatus'),''),
        'gw_health':       first(attrs.get('gwHealth'),''),
        'gw_type':         first(attrs.get('gatewayType'),''),
        'cctv_status':     first(attrs.get('cctvStatus'),    tele('cctvStatus'),''),
        'power_status':    first(attrs.get('powerStatus'),   tele('powerStatus'),''),
        'ups_status':      first(tele('upsStatus'),''),
        'recording_status':first(tele('recordingStatus'),''),

        # ── CAMERA ────────────────────────────────────────────────────────────
        'cam_total':       safe_int(first(tele('cameraCount'),  attrs.get('cameraCount'),0)),
        'cam_online':      safe_int(first(tele('cameraOnline'), attrs.get('cameraOnline'),0)),
        'cam_offline':     safe_int(first(tele('cameraOffline'),attrs.get('cameraOffline'),0)),
        'cam_dc_count':    safe_int(attrs.get('cameraDisconnectCount',0)),
        'cam_tamper_count':safe_int(attrs.get('cameraTamperCount',0)),
        'count_ch':        safe_int(attrs.get('count_CH',0)),
        'count_hdd':       safe_int(attrs.get('count_HDD',0)),
        'cam_link_status': json.dumps(to_json(attrs.get('cameraLinkStatus')) or {}),
        'low_dur_cameras': str(attrs.get('lowDurationCameras','') or ''),

        # ── SYSTEM HEALTH ─────────────────────────────────────────────────────
        'disk_pct':        safe_float(sh.get('disk',   tele('disk',0))),
        'cpu_pct':         safe_float(sh.get('cpu',    tele('cpu', 0))),
        'ram_pct':         safe_float(sh.get('ram',    tele('ram', 0))),
        'battery_voltage': safe_float(sh.get('battery_voltage', tele('battery_voltage',0))),
        'temperature':     safe_float(tele('temperature',0)),
        'uptime_sec':      safe_float(tele('uptime',0)),

        # ── GPS ───────────────────────────────────────────────────────────────
        'latitude':        safe_float(first(tele('arrLat'),  tele('latitude'),  0)),
        'longitude':       safe_float(first(tele('arrLon'),  tele('longitude'), 0)),

        # ── TELEMETRY ─────────────────────────────────────────────────────────
        'total_data_mb':   safe_float(tele('Total_Data_Usage',0)),
        'bas_downtime_min':safe_float(tele('BAS_Downtime_Minutes',0)),
        'nvr_downtime_min':safe_float(tele('NVR_Downtime_Minutes',0)),
        'fas_downtime_min':safe_float(tele('FAS_Downtime_Minutes',0)),
        'ias_downtime_min':safe_float(tele('IAS_Downtime_Minutes',0)),
        'acs_downtime_min':safe_float(tele('ACS_Downtime_Minutes',0)),
        'cavli_ontime':    safe_float(tele('cavlidata_ontime',0)),
        'sim_iccid':       first(tele('sim_iccid'),''),
        'sim_operator':    first(tele('sim_operator'),''),
        'signal_strength': safe_float(tele('signal_strength',0)),
        'network_type':    first(tele('network_type'),''),
        'ip_address':      first(tele('ip_address'),''),

        # ── SOFTWARE / OTA ────────────────────────────────────────────────────
        'sw_state':        first(tele('sw_state'),''),
        'sw_version':      first(tele('sw_version'), tele('target_sw_version'),''),
        'sw_title':        first(tele('target_sw_title'),''),
        'sw_tag':          first(tele('target_sw_tag'),''),
        'fw_version':      first(tele('fw_version'),''),
        'fw_state':        first(tele('fw_state'),''),

        # ── TIMESTAMPS ────────────────────────────────────────────────────────
        'last_update':     first(attrs.get('lastUpdate'),     tele('lastUpdate'),''),
        'last_connect':    epoch_ms(attrs.get('lastConnectTime')),
        'last_disconnect': epoch_ms(attrs.get('lastDisconnectTime')),
        'last_activity':   epoch_ms(attrs.get('lastActivityTime')),
        'inactive_since':  first(attrs.get('inactiveSince'),  tele('inactiveSince'),''),
        'inactive_reason': first(attrs.get('inactiveReason'), tele('inactiveReason'),''),
        'inactivity_alarm':epoch_ms(attrs.get('inactivityAlarmTime')),
        'bas_alarm_ts':    epoch_ms(attrs.get('basAlarmCreatedTime')),
        'fas_alarm_ts':    epoch_ms(attrs.get('fasAlarmCreatedTime')),
        'ias_alarm_ts':    epoch_ms(attrs.get('iasAlarmCreatedTime')),
        'gw_alarm_ts':     epoch_ms(attrs.get('gatewayAlarmCreatedTime')),
        'tls_alarm_ts':    epoch_ms(attrs.get('timeLockAlarmCreatedTime')),
        'cctv_alarm_ts':   epoch_ms(attrs.get('cctvAlarmCreatedTime')),

        # ── ALARM METADATA ────────────────────────────────────────────────────
        'alarm_flag':      attrs.get('alarmFlag'),
        'severity_attr':   first(attrs.get('severity'),''),
        'critical_flag':   attrs.get('critical'),
        'major_flag':      attrs.get('major'),
        'warning_flag':    attrs.get('warning'),
        'notification':    attrs.get('notification'),
        'care':            attrs.get('care'),

        # ── SUBSYSTEM COUNTS ──────────────────────────────────────────────────
        'total_sys_intrusion': attrs.get('Total System(Intrusion)',''),
        'total_sys_timelock':  attrs.get('Total System(Time Lock)',''),
        'faulty_intrusion':    attrs.get('Faulty Device(Intrusion)',''),
        'faulty_timelock':     attrs.get('Faulty Device(Time Lock)',''),
        'healthy_intrusion':   attrs.get('Healthy Device(Intrusion)',''),
        'healthy_timelock':    attrs.get('Healthy Device(Time Lock)',''),
        'inactive_intrusion':  attrs.get('Inactive Device(Intrusion)',''),
        'inactive_timelock':   attrs.get('Inactive Device(Time Lock)',''),
        'inactive_device_name':attrs.get('inactiveDeviceName',''),

        # ── USAGE HISTORY ─────────────────────────────────────────────────────
        'usage_history_json':  json.dumps(to_json(
            attrs.get('usage_history', attrs.get('usageHistory',[]))) or []),
        'usage_daily_json':    json.dumps(to_json(attrs.get('usage_daily','')) or []),
        'usage_last_7d_json':  json.dumps(to_json(attrs.get('usage_last_7_days','')) or []),
        'usage_last_15d_json': json.dumps(to_json(attrs.get('usage_last_15_days','')) or []),

        # ── RAW BLOBS ─────────────────────────────────────────────────────────
        'raw_subsystems':      json.dumps(to_json(attrs.get('subsystems')) or {}),
        'raw_event_metadata':  json.dumps(to_json(attrs.get('eventMetadata')) or {}),
        'raw_dexter_config':   json.dumps(to_json(attrs.get('dexter_config')) or {}),
        'raw_access_control':  json.dumps(to_json(attrs.get('accessControl')) or {}),
        'raw_tls_mili_time':   str(attrs.get('timeLockMiliTime','') or ''),
    }

    # Event flags
    for col, attr_key in EVENT_FLAG_MAP.items():
        row[col] = attrs.get(attr_key)

    # Per-channel camera histories
    row.update(ch_dc)
    row.update(ch_tp)

    rows.append(row)

device_df = pd.DataFrame(rows)
print(f'\n✅ device_df: {len(device_df)} rows × {len(device_df.columns)} columns')


⚙️  Building master DataFrame for 184 devices ...

✅ device_df: 184 rows × 222 columns


---
## Cell 10 — Fault Scoring Engine

In [24]:
def compute_gap_days(usage_json_str):
    try: history = json.loads(usage_json_str or '[]')
    except: return 0, None
    if not history: return 0, None
    try: history = sorted(history, key=lambda x: x.get('date',''))
    except: return 0, None
    max_s = cur = 0; ss = bs = None
    for e in history:
        try: cnt = float(e.get('count', e.get('value',-1)))
        except: cnt = -1
        if cnt == 0:
            cur += 1
            if cur == 1: ss = e.get('date','')
            if cur > max_s: max_s, bs = cur, ss
        else: cur = 0; ss = None
    return max_s, bs

def score_device(row):
    s, reasons = 0.0, []

    # Usage gap
    gap, gap_start = compute_gap_days(row.get('usage_history_json','[]'))
    if   gap >= 90: s += 50; reasons.append(f'GAP_{gap}d(+50)')
    elif gap >= 30: s += 40; reasons.append(f'GAP_{gap}d(+40)')
    elif gap >= 7:  s += 25; reasons.append(f'GAP_{gap}d(+25)')
    elif gap >= GAP_FAULT_DAYS: s += 12; reasons.append(f'GAP_{gap}d(+12)')

    # BAS downtime
    bd = safe_float(row.get('bas_downtime_min',0))
    if   bd >= 1440: s += 30; reasons.append(f'BAS_DT_{bd:.0f}m(+30)')
    elif bd >= 480:  s += 20; reasons.append(f'BAS_DT_{bd:.0f}m(+20)')
    elif bd >= 60:   s += 10; reasons.append(f'BAS_DT_{bd:.0f}m(+10)')

    # Zero data
    if safe_float(row.get('total_data_mb',-1)) == 0:
        s += 15; reasons.append('ZERO_DATA(+15)')

    # Inactive
    if row.get('inactive_since') and str(row['inactive_since']).strip() not in ('','null','None'):
        s += 20; reasons.append('INACTIVE(+20)')

    # NVR/DVR
    if is_fault(row.get('nvr_status')):
        s += 30; reasons.append(f'NVR={row["nvr_status"]}(+30)')
    elif is_active(row.get('ev_dvr_off')):
        s += 28; reasons.append('DVR_OFF(+28)')

    # HDD
    if is_fault(row.get('hdd_status')):
        s += 30; reasons.append(f'HDD={row["hdd_status"]}(+30)')
    elif is_active(row.get('ev_hdd_error')):
        s += 25; reasons.append('HDD_ERR(+25)')

    # Power / Gateway
    if is_active(row.get('ev_power_off')): s += 20; reasons.append('PWR_OFF(+20)')
    elif is_fault(row.get('gw_status')):   s += 15; reasons.append('GW_FAULT(+15)')

    # Camera disconnects
    dc = safe_int(row.get('cam_dc_count',0))
    if dc > 0: pts = min(dc*5,25); s += pts; reasons.append(f'CAM_DC={dc}(+{pts})')
    elif is_active(row.get('ev_cam_disconnect')): s += 15; reasons.append('CAM_DC_EV(+15)')

    # FAS
    if is_fault(row.get('fas_status')):
        s += 20; reasons.append('FAS_FAULT(+20)')
    elif is_active(row.get('ev_fas_off')): s += 20; reasons.append('FAS_OFF(+20)')
    elif is_active(row.get('ev_fas_fault')): s += 18; reasons.append('FAS_FLT_EV(+18)')

    # IAS
    if is_fault(row.get('ias_status')) or is_active(row.get('ev_ias_off')):
        s += 15; reasons.append('IAS_FAULT(+15)')

    # ACS / BAS / TLS
    if is_fault(row.get('acs_status')): s += 15; reasons.append('ACS_FAULT(+15)')
    if is_fault(row.get('bas_status')): s += 12; reasons.append('BAS_FAULT(+12)')
    if is_fault(row.get('tls_status')) or is_active(row.get('ev_tls_off')):
        s += 10; reasons.append('TLS_FAULT(+10)')

    # Battery
    if is_active(row.get('ev_battery_low')): s += 10; reasons.append('BATT_LOW(+10)')
    if is_active(row.get('ev_battery_reverse')): s += 8; reasons.append('BATT_REV(+8)')

    # SW state
    if str(row.get('sw_state','')).upper() in ('FAILED','FAILED_UPDATE','ERROR'):
        s += 10; reasons.append('FW_FAIL(+10)')

    # Disk / CPU
    disk = safe_float(row.get('disk_pct',0))
    cpu  = safe_float(row.get('cpu_pct',0))
    if   disk >= 90: s += 20; reasons.append(f'DISK_CRIT({disk:.0f}%)')
    elif disk >= 80: s += 15; reasons.append(f'DISK_HIGH({disk:.0f}%)')
    elif disk >= 75: s += 8;  reasons.append(f'DISK_WARN({disk:.0f}%)')
    if   cpu  >= 90: s += 10; reasons.append(f'CPU_HIGH({cpu:.0f}%)')

    score = round(min(s,100), 2)
    sev   = ('CRITICAL' if score>=70 else 'HIGH' if score>=45
             else 'MEDIUM' if score>=20 else 'HEALTHY')
    return score, sev, gap, gap_start or '', ' | '.join(reasons[:6]) or 'OK'

print('⚙️  Scoring ...')
device_df[['fault_score','severity','gap_days','gap_start','top_reasons']] = \
    device_df.apply(lambda r: pd.Series(score_device(r)), axis=1)
device_df = device_df.sort_values('fault_score', ascending=False).reset_index(drop=True)

print(f'\n✅ Scored {len(device_df)} devices\n')
counts = device_df['severity'].value_counts()
for sev in ['CRITICAL','HIGH','MEDIUM','HEALTHY']:
    n   = int(counts.get(sev,0))
    bar = '█' * (n*30//max(len(device_df),1))
    print(f'  {sev:<10} {n:>5}  {bar}')

print('\nTOP 15 by fault score:')
print(device_df[['bank_name','nbg_name','zo_name','branch_name',
                 'fault_score','severity','gap_days',
                 'nvr_status','bas_downtime_min']]
      .head(15).to_string(index=False))


⚙️  Scoring ...

✅ Scored 184 devices

  CRITICAL     109  █████████████████
  HIGH          35  █████
  MEDIUM         6  
  HEALTHY       34  █████

TOP 15 by fault score:
     bank_name    nbg_name       zo_name               branch_name  fault_score severity  gap_days nvr_status  bas_downtime_min
                  NBG EAST     ZO HOWRAH                 Branch TR        100.0 CRITICAL         0    Healthy               0.0
BANK OF BARODA ZO(Kolkata)    ZO_Kolkata     Branch_Shakuntal_Park        100.0 CRITICAL         0    Healthy               0.0
BANK OF BARODA ZO(Kolkata)    ZO KOLKATA   BRANCH_SURYA SEN STREET        100.0 CRITICAL         0   Inactive               0.0
 BANK OF INDIA    NBG EAST   ZO GUWAHATI           BRANCH AGARTALA        100.0 CRITICAL         0   Inactive               0.0
 BANK OF INDIA    NBG EAST    ZO BARASAT BRANCH BAGUIHATI TEGHORIA        100.0 CRITICAL         0    Healthy               0.0
 BANK OF INDIA      NBG MP    ZO KHANDWA              BRAN

---
## Cell 11 — Hierarchy Summary Tables (Bank / HO / NBG / ZO / Branch)

In [25]:
def agg_summary(group_cols):
    available = [c for c in group_cols if c in device_df.columns]
    return device_df.groupby(available, as_index=False, dropna=False).agg(
        devices          =('device_id',         'count'),
        avg_score        =('fault_score',         'mean'),
        max_score        =('fault_score',         'max'),
        critical         =('severity',            lambda x: (x=='CRITICAL').sum()),
        high             =('severity',            lambda x: (x=='HIGH').sum()),
        medium           =('severity',            lambda x: (x=='MEDIUM').sum()),
        healthy          =('severity',            lambda x: (x=='HEALTHY').sum()),
        gap_devices      =('gap_days',            lambda x: (x>=GAP_FAULT_DAYS).sum()),
        max_gap_days     =('gap_days',            'max'),
        total_data_mb    =('total_data_mb',       'sum'),
        bas_down_hrs     =('bas_downtime_min',    lambda x: round(x.sum()/60,1)),
        nvr_down_hrs     =('nvr_downtime_min',    lambda x: round(x.sum()/60,1)),
        fas_down_hrs     =('fas_downtime_min',    lambda x: round(x.sum()/60,1)),
        cam_dc_total     =('cam_dc_count',        'sum'),
    ).assign(avg_score=lambda d: d['avg_score'].round(1)
    ).sort_values('avg_score', ascending=False)

bank_df   = agg_summary(['bank_name'])
ho_df     = agg_summary(['bank_name','ho_name'])
nbg_df    = agg_summary(['bank_name','nbg_name'])
zo_df     = agg_summary(['bank_name','nbg_name','zo_name'])
ro_df     = agg_summary(['bank_name','zo_name','ro_name'])
branch_df = agg_summary(['bank_name','nbg_name','zo_name','branch_name'])

# Add child counts to bank summary
bank_df['unique_zones']    = (device_df.groupby('bank_name')['zo_name']
                              .nunique().reindex(bank_df['bank_name']).values)
bank_df['unique_branches'] = (device_df.groupby('bank_name')['branch_name']
                              .nunique().reindex(bank_df['bank_name']).values)

print(f'✅ Summary tables built:')
print(f'   bank_df   : {len(bank_df)} rows')
print(f'   ho_df     : {len(ho_df)} rows')
print(f'   nbg_df    : {len(nbg_df)} rows')
print(f'   zo_df     : {len(zo_df)} rows')
print(f'   ro_df     : {len(ro_df)} rows')
print(f'   branch_df : {len(branch_df)} rows')
print()
print('BANK LEVEL FAULT RANKING:')
print(bank_df[['bank_name','unique_zones','unique_branches','devices',
               'avg_score','critical','high','bas_down_hrs']]
      .to_string(index=False))


✅ Summary tables built:
   bank_df   : 22 rows
   ho_df     : 22 rows
   nbg_df    : 32 rows
   zo_df     : 41 rows
   ro_df     : 39 rows
   branch_df : 136 rows

BANK LEVEL FAULT RANKING:
                bank_name  unique_zones  unique_branches  devices  avg_score  critical  high  bas_down_hrs
             Dexter 4 CUS             1                1        1      100.0         1     0           0.0
        DEXTER RANCHI CUS             1                1        1      100.0         1     0           0.0
            SDF-RASP5 CUS             1                1        1      100.0         1     0           0.0
             LOHARDAGA CC             1                1        1       85.0         1     0           0.0
            BANK OF INDIA            10               98       98       78.8        70    26           0.0
           BANK OF BARODA             5               11       11       77.8        10     0           0.0
                                      4                5     

---
## Cell 12 — Key Discovery Report

In [26]:
kc = {}
for a in device_data:
    for k,v in a.items():
        if not k.startswith('_') and v not in (None,'','[]','{}',{},[]):
            kc[k] = kc.get(k,0) + 1

kd = sorted(kc.items(), key=lambda x: -x[1])
known = set(CLIENT_KEYS + SERVER_KEYS + TELEMETRY_KEYS)
new_k = [(k,n) for k,n in kd if k not in known and not k.startswith('tele_')]

key_disc_df = pd.DataFrame(
    [(k, n, round(100*n/max(len(device_data),1),1)) for k,n in kd],
    columns=['key','devices_with_value','coverage_pct']
)

print(f'{len(kd)} unique keys across {len(device_data)} devices')
print(f'{len(new_k)} new keys not in defined lists:')
for k,n in new_k[:30]:
    print(f'  {k:<55} {n:>5} ({100*n//max(len(device_data),1)}%)')


548 unique keys across 184 devices
278 new keys not in defined lists:
  bas_downtime_min                                          184 (100%)
  total_data_mb                                             184 (100%)
  audit_ts                                                  184 (100%)
  hierarchy_depth                                           184 (100%)
  fault_score                                               184 (100%)
  fault_severity                                            184 (100%)
  fault_reasons                                             184 (100%)
  gap_days                                                  184 (100%)
  full_path                                                 135 (73%)
  nbg_name                                                  134 (72%)
  bank_name                                                 130 (70%)
  zo_name                                                   129 (70%)
  branch_name                                               121 (65%)
  res       

---
## Cell 13 — Export Excel (13 Sheets) + Dashboard JSON + ML JSONL

In [27]:
from openpyxl.styles import PatternFill, Font, Alignment
from openpyxl.utils import get_column_letter
from openpyxl import load_workbook
import os

ts          = datetime.now().strftime('%Y%m%d_%H%M')
XLSX_PATH   = f'tb_audit_v9_{ts}.xlsx'
JSON_PATH   = 'dashboard_data.json'
JSONL_PATH  = 'ml_training_v9.jsonl'

FILLS = {'CRITICAL':PatternFill('solid',fgColor='FFCCCC'),
         'HIGH':    PatternFill('solid',fgColor='FFE5CC'),
         'MEDIUM':  PatternFill('solid',fgColor='FFFACC'),
         'HEALTHY': PatternFill('solid',fgColor='CCFFCC')}
HDR = PatternFill('solid',fgColor='1B3A5C')
HF  = Font(bold=True,color='FFFFFF')
HA  = Alignment(horizontal='center',vertical='center',wrap_text=True)

def style_ws(ws, sev_col=None):
    for c in ws[1]: c.fill=HDR; c.font=HF; c.alignment=HA
    ws.row_dimensions[1].height=28
    ws.freeze_panes='A2'
    for col in ws.columns:
        w=max((len(str(c.value or '')) for c in col),default=8)
        ws.column_dimensions[get_column_letter(col[0].column)].width=min(w+3,42)
    if sev_col:
        sc=next((c.column for c in ws[1] if str(c.value)==sev_col),None)
        if sc:
            for row in ws.iter_rows(min_row=2,min_col=sc,max_col=sc):
                for cell in row:
                    cell.fill=FILLS.get(str(cell.value),PatternFill())

# Drop raw/channel blob cols from main sheet
DROP = [c for c in device_df.columns
        if c.startswith('raw_') or c.startswith('camDC_') or c.startswith('camTP_')]
df_exp = device_df.drop(columns=DROP, errors='ignore')

# Hierarchy map sheet
hier_cols = ['bank_name','ho_name','nbg_name','lho_name','zo_name','ro_name',
             'co_name','rbo_name','branch_name','branch_code',
             'device_name','device_id','device_type',
             'full_path','hierarchy_depth',
             'branch_lat','branch_lon','branch_city','branch_state',
             'install_date','go_live_date','contract_type','sla_tier']
hier_df = device_df[[c for c in hier_cols if c in device_df.columns]].copy()

# ── EXCEL export ──────────────────────────────────────────────────────────────
with pd.ExcelWriter(XLSX_PATH, engine='openpyxl') as w:
    df_exp.to_excel(w,                                          sheet_name='All Devices',       index=False)
    df_exp[df_exp['fault_score']>=70].to_excel(w,               sheet_name='Critical',           index=False)
    df_exp[df_exp['gap_days']>0].sort_values('gap_days',ascending=False).to_excel(
                                                                w, sheet_name='Usage Gaps',       index=False)
    bank_df.to_excel(w,                                         sheet_name='Bank Summary',       index=False)
    ho_df.to_excel(w,                                           sheet_name='HO Summary',         index=False)
    nbg_df.to_excel(w,                                          sheet_name='NBG Summary',        index=False)
    zo_df.to_excel(w,                                           sheet_name='Zone Summary',       index=False)
    ro_df.to_excel(w,                                           sheet_name='RO Summary',         index=False)
    branch_df.to_excel(w,                                       sheet_name='Branch Summary',     index=False)
    hier_df.to_excel(w,                                         sheet_name='Hierarchy Map',      index=False)
    pd.DataFrame(customers).drop(columns=['_attrs'],errors='ignore').to_excel(
                                                                w, sheet_name='Customers (Banks)',index=False)
    pd.DataFrame([{k:v for k,v in a.items() if k!='_attrs'} for a in assets]
                ).to_excel(w,                                   sheet_name='Assets',             index=False)
    key_disc_df.to_excel(w,                                     sheet_name='Key Discovery',      index=False)

wb = load_workbook(XLSX_PATH)
SEV = {'All Devices','Critical','Usage Gaps'}
for sn in wb.sheetnames:
    style_ws(wb[sn], 'severity' if sn in SEV else None)
wb.save(XLSX_PATH)
print(f'✅ Excel: {XLSX_PATH} ({os.path.getsize(XLSX_PATH)//1024} KB) — {len(wb.sheetnames)} sheets')

# ── Dashboard JSON ────────────────────────────────────────────────────────────
def df2rec(d):
    return json.loads(d.to_json(orient='records', default_handler=str))

dashboard = {
    'generated_at': datetime.utcnow().isoformat()+'Z',
    'summary': {
        'total_devices':    int(len(device_df)),
        'total_banks':      int(device_df['bank_name'].nunique()),
        'total_zones':      int(device_df['zo_name'].nunique()),
        'total_branches':   int(device_df['branch_name'].nunique()),
        'critical': int((device_df['severity']=='CRITICAL').sum()),
        'high':     int((device_df['severity']=='HIGH').sum()),
        'medium':   int((device_df['severity']=='MEDIUM').sum()),
        'healthy':  int((device_df['severity']=='HEALTHY').sum()),
    },
    'devices':  df2rec(df_exp[[
        'bank_name','nbg_name','zo_name','branch_name','device_name','device_id',
        'fault_score','severity','top_reasons','gap_days',
        'nvr_status','hdd_status','fas_status','ias_status','acs_status',
        'bas_status','tls_status','gw_status','cctv_status',
        'disk_pct','cpu_pct','ram_pct','total_data_mb','bas_downtime_min',
        'sw_state','sw_version','full_path','hierarchy_depth',
        'branch_lat','branch_lon',
    ]].head(1000)),
    'banks':    df2rec(bank_df),
    'zones':    df2rec(zo_df),
    'branches': df2rec(branch_df),
}
with open(JSON_PATH,'w') as f:
    json.dump(dashboard, f, default=str, indent=2)
print(f'✅ Dashboard JSON: {JSON_PATH} ({os.path.getsize(JSON_PATH)//1024} KB)')

# ── ML JSONL ──────────────────────────────────────────────────────────────────
EVENT_COLS = [c for c in device_df.columns if c.startswith('ev_')]
STATUS_COLS = ['nvr_status','hdd_status','fas_status','ias_status',
               'acs_status','bas_status','tls_status','gw_status',
               'cctv_status','device_status','sw_state']

written = skipped = 0
with open(JSONL_PATH,'w',encoding='utf-8') as f:
    for _, row in device_df.iterrows():
        parts = []
        if row.get('bank_name'):   parts.append(f'bank:{row["bank_name"]}')
        if row.get('nbg_name'):    parts.append(f'nbg:{row["nbg_name"]}')
        if row.get('zo_name'):     parts.append(f'zo:{row["zo_name"]}')
        if row.get('branch_name'): parts.append(f'branch:{row["branch_name"]}')
        if row.get('nvr_model'):   parts.append(f'nvr:{row["nvr_model"]}')
        gap = safe_int(row.get('gap_days',0))
        if gap > 0: parts.append(f'usage_gap_days:{gap}')
        for col in STATUS_COLS:
            v = str(row.get(col,'')).strip()
            if v and v.lower() not in ('','null','none','nan','0'):
                parts.append(f'{col}:{v}')
        active_evs = [c.replace('ev_','') for c in EVENT_COLS
                      if is_active(row.get(c))]
        if active_evs: parts.append(f'events:{",".join(active_evs)}')
        if safe_float(row.get('disk_pct',0)) > 0:
            parts.append(f'disk_pct:{row["disk_pct"]:.0f}')
        if safe_float(row.get('bas_downtime_min',0)) > 0:
            parts.append(f'bas_dt_min:{row["bas_downtime_min"]:.0f}')
        if safe_float(row.get('total_data_mb',0)) >= 0:
            parts.append(f'data_mb:{row["total_data_mb"]:.0f}')
        if safe_int(row.get('cam_dc_count',0)) > 0:
            parts.append(f'cam_dc:{row["cam_dc_count"]}')
        if row.get('hierarchy_depth'):
            parts.append(f'hier_depth:{row["hierarchy_depth"]}')
        if len(parts) < 3: skipped += 1; continue
        f.write(json.dumps({
            'input':  ' '.join(parts),
            'output': f'fault_class:{row["severity"]} fault_score:{row["fault_score"]:.0f} gap_days:{gap} reasons:{row["top_reasons"]}',
            '_meta':  {'device_id':row.get('device_id',''),
                       'bank':row.get('bank_name',''),'score':row.get('fault_score',0)},
        }, ensure_ascii=False) + '\n')
        written += 1

print(f'✅ ML JSONL: {JSONL_PATH} — {written} examples ({skipped} skipped)')
print(f'\n📊 FINAL SUMMARY:')
print(f'   Devices    : {len(device_df)}')
print(f'   Banks      : {device_df["bank_name"].nunique()}')
print(f'   Zones      : {device_df["zo_name"].nunique()}')
print(f'   Branches   : {device_df["branch_name"].nunique()}')
print(f'   Columns    : {len(device_df.columns)}')
for sev in ['CRITICAL','HIGH','MEDIUM','HEALTHY']:
    print(f'   {sev:<10}: {int((device_df["severity"]==sev).sum())}')


✅ Excel: tb_audit_v9_20260512_1743.xlsx (338 KB) — 13 sheets
✅ Dashboard JSON: dashboard_data.json (275 KB)
✅ ML JSONL: ml_training_v9.jsonl — 171 examples (13 skipped)

📊 FINAL SUMMARY:
   Devices    : 184
   Banks      : 22
   Zones      : 20
   Branches   : 122
   Columns    : 227
   CRITICAL  : 109
   HIGH      : 35
   MEDIUM    : 6
   HEALTHY   : 34


---
## Cell 14 — Write Back to ThingsBoard (Optional)

In [28]:
from tqdm import tqdm as _tqdm

WRITE_BACK = True   # ← set True when ready

if not WRITE_BACK:
    print('ℹ️  Skipped. Set WRITE_BACK = True to push fault_score to TB.')
else:
    errors = []
    for _, row in _tqdm(device_df.iterrows(), total=len(device_df), desc='Writing'):
        dev_id = row['device_id']
        if not dev_id: continue
        payload = {
            'fault_score':      float(row['fault_score']),
            'fault_severity':   str(row['severity']),
            'fault_reasons':    str(row['top_reasons']),
            'gap_days':         int(row['gap_days']),
            'bas_downtime_min': float(row.get('bas_downtime_min',0)),
            'total_data_mb':    float(row.get('total_data_mb',0)),
            'bank_name':        str(row.get('bank_name','')),
            'nbg_name':         str(row.get('nbg_name','')),
            'zo_name':          str(row.get('zo_name','')),
            'branch_name':      str(row.get('branch_name','')),
            'full_path':        str(row.get('full_path','')),
            'hierarchy_depth':  int(row.get('hierarchy_depth',0)),
            'audit_ts':         datetime.now().strftime('%Y-%m-%d %H:%M'),
        }
        url = f'{TB_HOST}/api/plugins/telemetry/DEVICE/{dev_id}/attributes/SERVER_SCOPE'
        try:
            r = session.post(url, headers=AUTH_HEADERS, json=payload, timeout=15)
            if r.status_code not in (200,201):
                errors.append(f'{row["device_name"]}: HTTP {r.status_code}')
        except Exception as e:
            errors.append(f'{row["device_name"]}: {e}')
        time.sleep(REQUEST_DELAY)
    print(f'Written: {len(device_df)-len(errors)}/{len(device_df)}')
    if errors: [print(f'  ⚠️  {e}') for e in errors[:10]]
    print('In TB: Device → Attributes → SERVER_SCOPE → fault_score / full_path')


Writing: 100%|██████████| 184/184 [00:19<00:00,  9.25it/s]

Written: 184/184
In TB: Device → Attributes → SERVER_SCOPE → fault_score / full_path
